# Phase 1: Test on assembly_1

## Section 1 - Purpose and protocol

This notebook measures the Phase 1 perception once, on the test recording assembly_1.

The method is fixed and nothing is chosen here. C-RADIOv4-SO400M is used frozen, with input 1024x1024. Patches are classified by V2, the nearest labelled patch of the support set of 10 frames, with background margin 0. The prototypes are loaded from the file saved by the pipeline at v1.9-pipeline (outputs/phase_1_perception/patch_prototypes.npz).

The 32 validation frames of assembly_2 were used to choose the method and the support set. Their values are therefore optimistic: an estimate made on data used for selection is biased (Cawley and Talbot 2010, Sect. 5). The test frames give the estimate that no choice has seen.

assembly_1 was never used for any choice. It was inspected once, qualitatively, in v1-baseline, before any method was chosen. At that time only f0073 was annotated.

The work follows a fixed order, and each step is committed before the next. First the event log of assembly_1 (configs/assembly_1_events.json). Then the rules and the list of test frames (Section 3). Then the annotation of these frames (Section 4) and the measurement plan (Section 5). Only then the single execution (Sections 6 and 7). No result of this notebook changes the method, the support set or the frames.

assembly_1 follows the same procedure as assembly_2, with the pieces in the same order, the same operator and the same environment. The test therefore measures perception on a new recording of the same procedure. It does not measure a different order, operator or scene.

------------------------------------------------------------------------------------------------------------------------------------------

References Section 1:

Cawley and Talbot 2010, Sect. 5: training and model selection must be treated together and never performed separately at any point of the evaluation; protocols that do not respect this give optimistically biased performance estimates.

Pipeline (phase_1_perception.ipynb), Section 6.4: the V2 prototypes saved with their classes and metadata.

Support-set study on V2, Section 3: the rules used to choose the validation frames, adapted here in Section 3.

------------------------------------------------------------------------------------------------------------------------------------------

## Section 2 - Configuration

All paths, classes, settings and test frames in one place.

The prototypes are not rebuilt: they are loaded from the file saved by the pipeline (Section 6), so this notebook reads no reference frame. The values of MIN_LABEL_COVERAGE and MARGIN written here are those of the pipeline; Section 6 checks them against the metadata saved with the prototypes.

The test frames are listed in 2.6, with the moment and the piece of each one. Section 3 recomputes them from the event log with the rules, and checks that the two lists are equal. This is the only notebook that reads frames of assembly_1 for evaluation, so frame_path accepts only them.

In [1]:
# 2.1 Dependencies: install once per environment with pip install -r requirements.txt (README, Setup)

In [2]:
# 2.2 Imports
import json
import sys
import time
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import transformers

In [3]:
# 2.3 Paths and shared module
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))   # makes src/perception importable from notebooks/

MODEL_DIR = PROJECT_ROOT / "models" / "c_radio_weights"
FRAMES_DIR = PROJECT_ROOT / "data" / "extracted_frames"
EVENTS_PATH = PROJECT_ROOT / "configs" / "assembly_1_events.json"                       # event log of the test video
PROTOTYPES_PATH = PROJECT_ROOT / "outputs" / "phase_1_perception" / "patch_prototypes.npz"  # saved by the pipeline, read only
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "phase_1_test"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {"MODEL_DIR": MODEL_DIR, "FRAMES_DIR": FRAMES_DIR, "EVENTS_PATH": EVENTS_PATH,
                   "PROTOTYPES_PATH": PROTOTYPES_PATH, "SRC_DIR": SRC_DIR}.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"[{status}] {name}: {path}")

# Shared code (src/perception), used by every Phase 1 notebook
from perception.features import FeatureExtractor
from perception.labels import sequence_of, check_annotation, rasterize_annotation, patch_label_grid
from perception.prototypes import load_patch_prototypes, sort_by_class, class_similarities_gpu, assign_labels
from perception.metrics import (confusion_matrix, frame_iou, frame_miou, bootstrap_miou, bootstrap_class_iou,
                                percentile_interval, MERGED_NAMES)
from perception.visualization import label_grid_to_rgb, render_overlay

[OK] MODEL_DIR: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\models\c_radio_weights
[OK] FRAMES_DIR: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\data\extracted_frames
[OK] EVENTS_PATH: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\configs\assembly_1_events.json
[OK] PROTOTYPES_PATH: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\outputs\phase_1_perception\patch_prototypes.npz
[OK] SRC_DIR: c:\Users\ADV\Universita\Magistrale\Tesi\vad-project\src


In [4]:
# 2.4 Official class list, defined once in src/perception/classes.py and shared by all notebooks
from perception.classes import CLASS_NAMES, HAND_LABEL, PIECE_LABELS, LEGACY_LABELS, CLASS_COLORS

print(f"{len(CLASS_NAMES)} classes: {CLASS_NAMES}")

12 classes: ['background', 'body', 'base', 'hat', 'eyes', 'ear', 'nose', 'moustache', 'mouth', 'arm_1', 'arm_2', 'hand']


In [5]:
# 2.5 Settings
# Values of the pipeline (phase_1_perception.ipynb, Section 2.5); Section 6 checks them against the saved metadata.

# --- Feature extraction (Section 7) ---
DEVICE = "cuda"
DTYPE = torch.bfloat16

# --- Patch labelling of the ground truth (Section 4) ---
MIN_LABEL_COVERAGE = 0.5        # a patch gets a class label if that class covers at least half of it (pipeline, Section 5)

# --- Classification (Section 7) ---
MARGIN = 0.0                    # background margin of V2 (margin study, Section 9)

# --- Figures (Sections 4 and 7) ---
MIN_BLOB_PATCHES = 5            # smallest blob that gets a text label in the overlays, as in the pipeline

# --- Reproducibility ---
RANDOM_SEED = 0
np.random.seed(RANDOM_SEED)
_ = torch.manual_seed(RANDOM_SEED)

In [6]:
# 2.6 Test frames
# Test: assembly_1 only. This notebook reads no reference or validation frame (prototypes are loaded, Section 6).
TEST_SEQUENCE = "assembly_1"


def frame_path(frame, suffix=".jpg"):
    """Path of a test frame or of its annotation. Refuses frames of any other sequence."""
    assert sequence_of(frame) == TEST_SEQUENCE, f"{frame} is not a test frame"
    return FRAMES_DIR / f"{frame}{suffix}"


# Test frames in time order, chosen by the rules of Section 3 from configs/assembly_1_events.json.
# moment = table, pick or mount; piece = the piece being picked or mounted (for table, None).
_TEST = [
    (2,  "table", None,             "pieces loose; base outside the frame"),
    (4,  "pick",  "base",           "base in hand"),
    (5,  "mount", "base",           "base at the work position"),
    (8,  "table", None,             "base alone at the work position"),
    (11, "pick",  "body",           "body in hand above the base"),
    (12, "mount", "body",           "body on the base"),
    (16, "table", None,             "body on base, mouth not yet picked"),
    (18, "pick",  "mouth",          "mouth in hand"),
    (19, "mount", "mouth",          "pin of the mouth at the slot"),
    (25, "table", None,             "mouth mounted"),
    (27, "pick",  "moustache",      "moustache picked"),
    (29, "pick",  "nose",           "nose picked, moustache in the other hand"),
    (31, "mount", "nose",           "nose threaded into the moustache, in hand"),
    (33, "mount", "nose+moustache", "nose and moustache at the slot"),
    (37, "table", None,             "nose and moustache mounted"),
    (39, "pick",  "eyes",           "eyes picked"),
    (40, "mount", "eyes",           "eyes at the slot"),
    (43, "table", None,             "face complete, ears loose"),
    (45, "pick",  "ear",            "ear (image right) in hand"),
    (47, "mount", "ear",            "ear (image right) at the head"),
    (48, "pick",  "ear",            "ear (image left) under the hand"),
    (52, "mount", "ear",            "ear (image left) being pushed in"),
    (54, "table", None,             "ears mounted, arms loose"),
    (55, "pick",  "arm_1",          "arm_1 just picked"),
    (57, "mount", "arm_1",          "arm_1 in its hole"),
    (60, "pick",  "arm_2",          "arm_2 just picked"),
    (62, "mount", "arm_2",          "arm_2 in its hole"),
    (65, "table", None,             "arms mounted, hat loose"),
    (67, "pick",  "hat",            "hat in hand"),
    (68, "mount", "hat",            "hat on the head"),
    (73, "table", None,             "figure complete, no hands"),
]
TEST_FRAMES = [{"frame": f"{TEST_SEQUENCE}_f{index:04d}", "index": index, "moment": moment, "piece": piece, "note": note}
               for index, moment, piece, note in _TEST]

for item in TEST_FRAMES:
    item["image_path"], item["json_path"] = frame_path(item["frame"]), frame_path(item["frame"], ".json")
    assert item["image_path"].exists(), f"Missing image: {item['frame']}"
indices = [item["index"] for item in TEST_FRAMES]
assert indices == sorted(set(indices)), "Test frames must be unique and in time order"

print(f"{len(TEST_FRAMES)} test frames of {TEST_SEQUENCE}")
display(pd.Series([item["moment"] for item in TEST_FRAMES], name="frames").value_counts()
        .reindex(["table", "pick", "mount"]).to_frame())

31 test frames of assembly_1


,count
frames,
table,9
pick,11
mount,11


Observation 2.6: The test set has 31 frames of assembly_1: 9 at the table, 11 at a pick and 11 at a mounting. Validation has 32 frames of assembly_2 (10, 11 and 11). There is one table frame fewer because no pause separates the two arms in assembly_1 (Section 3).

In [7]:
# 2.7 Environment
print(f"Python       : {platform.python_version()}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

Python       : 3.10.0
PyTorch      : 2.5.1+cu121
Transformers : 5.17.0
CUDA         : True
GPU          : NVIDIA GeForce RTX 4060 Laptop GPU


------------------------------------------------------------------------------------------------------------------------------------------

References Section 2: none. The settings are those of the pipeline (phase_1_perception.ipynb, Section 2.5); the test frames are chosen by the rules of Section 3.

------------------------------------------------------------------------------------------------------------------------------------------

## Section 3 - Test frames

This section fixes the test frames with rules, before any frame is annotated and before any prediction is made.

The rules are those of the support-set study on V2 (Section 3), applied to the event log of assembly_1 (configs/assembly_1_events.json). The frames lie on the grid of extracted frames: frame fNNNN is video frame 30 x NNNN. Each frame belongs to one of three moments.

Pick. The first extracted frame from the start of the contact that ends with the piece lifted, and before the insert of that piece. The start of the contact is the touch. When the touch happens outside the frame, it is the first frame where the piece is seen in hand (first_seen). This is the only change to the rule, and it is needed for base, body and mouth, which are picked outside the frame.

Mount. The first extracted frame from the insert of the piece, and before the hands leave the figure. A piece joined to another piece in the hand, like the nose threaded into the moustache, counts as a mounting.

Table. In each pause between two contacts, the extracted frame closest to the centre of the pause. Hands may be in view. Contacts start as in the pick rule and end when the hands leave the figure. A pick that is put back would end a pause like any other pick, and a touch without lifting gives no table frame; assembly_1 has neither, so the code accepts their absence. The last pause ends when the head turns away from the figure. The clause of the validation set on the 6 old frames does not apply, since assembly_1 has none.

The event log and these rules were committed before the frames were annotated. A chosen frame is never replaced because of how it looks.

The event log was corrected twice after its first commit, both times on the raw video and before any prediction. The insert of the hat was moved to 2037, since its first contact is hidden by the peg and is present at f0068. During the annotation, the insert of the ear on the image right was moved to 1391, since it is not yet in contact at f0046 (1380) and is in place by 1402. Each correction has its own commit. These are corrections of when a contact happens, not choices of a frame: the frames are always recomputed by the same rules.

Cell 3.1 recomputes the frames from the log and checks that they are exactly the list of Section 2.6.


In [8]:
# 3.1 The three rules recomputed from the event log, and compared with the list of Section 2.6
EVENT_LOG = json.loads(EVENTS_PATH.read_text())
EVENTS, NOTES = EVENT_LOG["events"], EVENT_LOG["notes"]
GRID = 30                                    # one extracted frame every 30 video frames (extract_frames.py)
LAST_INDEX = 78                              # last extracted frame of assembly_1 (2348 video frames)


def notes_with(text):
    """Frame intervals of the notes whose description contains text (possibly none)."""
    return [note["frames"] for note in NOTES if text in note["what"]]


def contact_start(event):
    """Start of the contact: the touch, or the first frame the piece is seen in hand when the touch is outside the frame."""
    return event["touch"] if event["touch"] is not None else event["first_seen"]


def first_frame(start, end):
    """First extracted frame at or after start and before end, or None if the interval contains none."""
    k = -(-start // GRID)                    # ceiling division
    return k if GRID * k < end else None


JOINS = notes_with("threaded into")          # pieces joined in hand: [first contact, fully joined]
PUT_BACK = notes_with("put back")            # pieces picked and put back: they end a pause like a pick
NO_LIFT = notes_with("without lifting")      # touches without lifting: no table frame inside them
(HEAD_TURN,) = notes_with("head turns")      # the last pause ends when the head turns away
END_OF_ASSEMBLY = HEAD_TURN[0]

# Pick and mount: the frames the rules give for every event of the log
rows = []
for e in EVENTS:
    piece = e["piece"] + (f" ({e['side']})" if "side" in e else "")
    rows.append({"moment": "pick", "piece": piece, "window": f"{contact_start(e)}-{e['insert']}",
                 "index": first_frame(contact_start(e), e["insert"])})
    rows.append({"moment": "mount", "piece": piece, "window": f"{e['insert']}-{e['hands_off']}",
                 "index": first_frame(e["insert"], e["hands_off"])})
for start, end in JOINS:
    rows.append({"moment": "mount", "piece": "join in hand", "window": f"{start}-{end}", "index": first_frame(start, end)})
events_table = pd.DataFrame(rows)
events_table["frame"] = events_table["index"].map(lambda k: "-" if pd.isna(k) else f"f{int(k):04d}")
display(events_table.drop(columns="index"))

# Table: pauses between contacts. Contacts that follow without a gap (start = previous end + 1) leave no pause.
busy = sorted([(contact_start(e), e["hands_off"]) for e in EVENTS] + [tuple(p) for p in PUT_BACK])
merged = []
for start, end in busy:
    if merged and start <= merged[-1][1] + 1:
        merged[-1] = (merged[-1][0], max(merged[-1][1], end))
    else:
        merged.append((start, end))
pauses, previous_end = [], -1
for start, end in merged:
    pauses.append((previous_end + 1, start - 1))
    previous_end = end
pauses.append((previous_end + 1, END_OF_ASSEMBLY - 1))

rows, expected = [], {"pick": set(), "mount": set(), "table": set()}
for start, end in pauses:
    grid = [k for k in range(start // GRID, end // GRID + 1)
            if start <= GRID * k <= end and not any(a <= GRID * k <= b for a, b in NO_LIFT)]
    centre = (start + end) / 2
    chosen = min(grid, key=lambda k: abs(GRID * k - centre)) if grid else None   # on a tie, the earlier frame
    if chosen is not None:
        expected["table"].add(chosen)
    rows.append({"pause": f"{start}-{end}", "centre": centre, "extracted frames": len(grid),
                 "chosen": "-" if chosen is None else f"f{chosen:04d}"})
display(pd.DataFrame(rows))

for row in events_table.itertuples():
    if not pd.isna(row.index):
        expected[row.moment].add(int(row.index))
listed = {moment: {item["index"] for item in TEST_FRAMES if item["moment"] == moment} for moment in expected}
for moment in ["table", "pick", "mount"]:
    print(f"{moment:5s}: {len(expected[moment])} from the log, {len(listed[moment])} listed in 2.6, "
          f"equal: {expected[moment] == listed[moment]}")
missing = events_table[events_table["frame"] == "-"]
print(f"Events without an extracted frame in their window: {len(missing)}")
assert all(expected[m] == listed[m] for m in expected), "The list of Section 2.6 does not follow the rules"
assert all(0 <= k <= LAST_INDEX for m in expected for k in expected[m])

,moment,piece,window,frame
0,pick,base,112-138,f0004
1,mount,base,138-164,f0005
2,pick,body,304-345,f0011
3,mount,body,345-400,f0012
4,pick,mouth,538-556,f0018
5,mount,mouth,556-680,f0019
6,pick,moustache,796-980,f0027
7,mount,moustache,980-1044,f0033
8,pick,nose,850-980,f0029
9,mount,nose,980-1044,f0033


,pause,centre,extracted frames,chosen
0,0-111,55.5,4,f0002
1,165-303,234.0,5,f0008
2,401-537,469.0,4,f0016
3,681-795,738.0,4,f0025
4,1045-1155,1100.0,4,f0037
5,1235-1337,1286.0,3,f0043
6,1429-1433,1431.0,0,-
7,1575-1643,1609.0,2,f0054
8,1885-1992,1938.5,4,f0065
9,2137-2233,2185.0,3,f0073


table: 9 from the log, 9 listed in 2.6, equal: True
pick : 11 from the log, 11 listed in 2.6, equal: True
mount: 11 from the log, 11 listed in 2.6, equal: True
Events without an extracted frame in their window: 0


Observation 3.1: The rules applied to the event log give exactly the 31 frames of Section 2.6: 9 table, 11 pick and 11 mount frames. Every event of the log has an extracted frame in its window.

For base, body and mouth the pick window starts at first_seen (112, 304 and 538), since they are picked outside the frame. The nose and the moustache share the mount frame f0033, as they enter the slot together; the second mount of that pair is the join in hand, f0031 (902-958).

The log gives 10 pauses. Nine contain at least one extracted frame and give a table frame. The pause between the two ears, 1429-1433, is shorter than one extracted frame and gives none, like two pauses of assembly_2. Between the two arms there is no pause: the hand leaves arm_1 at 1781 and touches arm_2 at 1782. This is why the test set has one table frame fewer than the validation set.

The last table frame, f0073, falls in the final pause (2137-2233). It is the only frame of assembly_1 annotated before this study (v1-baseline); here it is chosen by the rule, and its annotation is checked in Section 4 like the others.

After the correction of its insert, the mount frame of the ear on the image right is f0047 instead of f0046.

------------------------------------------------------------------------------------------------------------------------------------------

References Section 3:

Support-set study on V2, Section 3: the rules for pick, mount and table frames, and the check that recomputes the frames from the event log.

configs/assembly_1_events.json: touch, first_seen, insert and hands_off of every piece, the join in hand and the head turn.

extract_frames.py: one frame every 30 video frames, named by its position in the extracted sequence.

------------------------------------------------------------------------------------------------------------------------------------------